In [1]:
import numpy as np 
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
import os
import scipy.io as sio
import pingouin as pg

In [2]:
child_data_dir = "../children_data"
adult_data_dir = "../adult_data"

In [3]:
# get averages of any column across all participants in directory

def get_averages(data_directory, column_name, num_rows_to_process):
    files_in_dir = [f for f in os.listdir(data_directory) if f.endswith('.mat')]
    num_files = len(files_in_dir)

    S_matrix = np.full((num_rows_to_process, num_files), np.nan)

    for i, filename in enumerate(files_in_dir):
        file_path = os.path.join(data_directory, filename)
        mat_contents = sio.loadmat(file_path)
        data_column = mat_contents[column_name]

        squeezed_data = data_column.squeeze()
        processed_data = squeezed_data.flatten() if hasattr(squeezed_data, 'flatten') else np.array([squeezed_data])
        
        elements_to_extract = min(len(processed_data), num_rows_to_process)
        S_matrix[:elements_to_extract, i] = processed_data[:elements_to_extract]
            
    N = np.sum(~np.isnan(S_matrix), axis=1)
    std_dev = np.nanstd(S_matrix, axis=1, ddof=1)
    
    sem = np.zeros(num_rows_to_process)
    valid_n_for_sem = N > 1
    sem[valid_n_for_sem] = std_dev[valid_n_for_sem] / np.sqrt(N[valid_n_for_sem])

    S_matrix_for_mean = np.copy(S_matrix)
    S_matrix_for_mean[np.isnan(S_matrix_for_mean)] = 0 
    averages = np.mean(S_matrix_for_mean, axis=1)
            
    return averages, sem


def get_all_participant_values(data_dir, column, phase_idx):
    values = []
    for filename in os.listdir(data_dir):
        if filename.endswith('.mat'):
            mat = sio.loadmat(os.path.join(data_dir, filename))
            col = mat[column].squeeze()
            # Defensive: handle both 1D and 2D
            if col.ndim > 0 and len(col) > phase_idx:
                values.append(col[phase_idx])
    return np.array(values)

def get_group_data(column, n):
    child_mean, child_sem = get_averages(child_data_dir, column, n)
    adult_mean, adult_sem = get_averages(adult_data_dir, column, n)
    return child_mean[:3], child_sem[:3], adult_mean[:3], adult_sem[:3]

In [4]:
get_all_participant_values(child_data_dir, 'meanMT', 0)

array([0.8905    , 1.0164    , 1.0788    , 0.971     , 1.0929    ,
       0.9059    , 0.8487    , 1.1185    , 1.00777778, 0.8638    ,
       0.8114    ])

In [5]:
sio.loadmat(os.path.join(child_data_dir, 'VML_MEG_002_Final_Results.mat'))['MT'].shape

(130, 1)

In [6]:
sio.loadmat(os.path.join(child_data_dir, 'VML_MEG_002_Final_Results.mat')).keys()

dict_keys(['__header__', '__version__', '__globals__', 'AA', 'ACC', 'DIRECTION', 'DV', 'EA', 'IDE', 'KEEP', 'MT', 'OUTLIER', 'PA', 'PV', 'RT', 'TV', 'meanMT', 'meanRT'])

In [7]:
df = pd.DataFrame({
    'group': ['children'] * 11 + ['adults'] * 13,
    'baseline_meanMT': np.append(get_all_participant_values(child_data_dir, 'meanMT', 0), get_all_participant_values(adult_data_dir, 'meanMT', 0)),
    'early_learning_meanMT': np.append(get_all_participant_values(child_data_dir, 'meanMT', 1), (get_all_participant_values(adult_data_dir, 'meanMT', 1))),
    'late_learning_meanMT': np.append(get_all_participant_values(child_data_dir, 'meanMT', 2), get_all_participant_values(adult_data_dir, 'meanMT', 2)),
})

In [8]:
df

,group,baseline_meanMT,early_learning_meanMT,late_learning_meanMT
0,children,0.890500,1.074379,1.080207
1,children,1.016400,0.971120,0.921480
2,children,1.078800,1.082500,1.066533
3,children,0.971000,1.188467,1.135759
4,children,1.092900,1.122100,1.100300
5,children,0.905900,0.893318,0.843217
6,children,0.848700,1.078367,1.053828
7,children,1.118500,1.128345,1.182929
8,children,1.007778,1.121333,1.110370
9,children,0.863800,0.932667,1.064833


In [12]:
df_mt_long = pd.melt(
    df,
    id_vars=['group'],
    value_vars=['baseline_meanMT', 'early_learning_meanMT', 'late_learning_meanMT'],
    var_name='phase',
    value_name='meanMT'
)

df_mt_long['subject'] = np.tile(np.arange(len(df)), 3)

aov_mt = pg.mixed_anova(
    dv='meanMT',
    within='phase',
    between='group',
    subject='subject',
    data=df_mt_long
)

df_mt_long
# print(aov_mt)

,group,phase,meanMT,subject
0,children,baseline_meanMT,0.890500,0
1,children,baseline_meanMT,1.016400,1
2,children,baseline_meanMT,1.078800,2
3,children,baseline_meanMT,0.971000,3
4,children,baseline_meanMT,1.092900,4
...,...,...,...,...
67,adults,late_learning_meanMT,0.968133,19
68,adults,late_learning_meanMT,1.102567,20
69,adults,late_learning_meanMT,1.091133,21
70,adults,late_learning_meanMT,1.025767,22


In [ ]:
df_rt = pd.DataFrame({
    'group': ['children'] * 11 + ['adults'] * 13,
    'baseline_meanRT': np.append(get_all_participant_values(child_data_dir, 'meanRT', 0), get_all_participant_values(adult_data_dir, 'meanRT', 0)),
    'early_learning_meanRT': np.append(get_all_participant_values(child_data_dir, 'meanRT', 1), get_all_participant_values(adult_data_dir, 'meanRT', 1)),
    'late_learning_meanRT': np.append(get_all_participant_values(child_data_dir, 'meanRT', 2), get_all_participant_values(adult_data_dir, 'meanRT', 2)),
})

df_rt_long = pd.melt(
    df_rt,
    id_vars=['group'],
    value_vars=['baseline_meanRT', 'early_learning_meanRT', 'late_learning_meanRT'],
    var_name='phase',
    value_name='meanRT'
)

df_rt_long['subject'] = np.tile(np.arange(len(df)), 3)

aov_rt = pg.mixed_anova(
    dv='meanRT',
    within='phase',
    between='group',
    subject='subject',
    data=df_rt_long
)
print(aov_rt)

        Source        SS  DF1  DF2        MS         F     p-unc       np2  \
0        group  0.225317    1   22  0.225317  8.947897  0.006729  0.289128   
1        phase  0.001869    2   44  0.000934  0.758509  0.474390  0.033329   
2  Interaction  0.000086    2   44  0.000043  0.034878  0.965750  0.001583   

        eps  
0       NaN  
1  0.809824  
2       NaN  
